# 05 · Window functions (Spark)

A **window function** computes a value from a group of rows (the *window*) but **keeps every row**
(GROUP BY would squash them into one).

```
function() OVER (PARTITION BY <groups> ORDER BY <sort> ROWS BETWEEN ... )
```
- `PARTITION BY` = restart for each group (like GROUP BY, but rows stay)
- `ORDER BY` = the order inside the group (needed for ranking, running totals, LAG/LEAD)
- frame (`ROWS BETWEEN ...`) = which rows around the current row to use

You **cannot** use a window function in WHERE. Wrap it in a subquery/CTE and filter outside.
BigQuery (and Oracle 23ai) also has `QUALIFY`. Spark SQL has no QUALIFY.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. ROW_NUMBER vs RANK vs DENSE_RANK  ·  🟢 easy

For department 2 (Sales), show `name`, `salary` and three columns ordered by salary (high first):
`rn` (ROW_NUMBER, ties broken by name), `rnk` (RANK), `drnk` (DENSE_RANK). Look at the 90,000 tie: what does each function do?

*Expected result: 6 rows · columns: `name`, `salary`, `rn`, `rnk`, `drnk`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q2. Top 2 salary levels per department  ·  🟡 medium

For each department (skip NULL dept_id) show employees whose salary is in the **top 2 different salary values** of that department. Show `dept_id`, `name`, `salary`.

*Expected result: 15 rows · columns: `dept_id`, `name`, `salary`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q3. Third highest salary  ·  🟡 medium

Show the **3rd highest different** salary in the company as `third_highest`.

*Expected result: 1 row · columns: `third_highest`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q4. Running total of revenue  ·  🟡 medium

For **completed** orders in 2024 (each order once), show `order_date`, `daily_revenue` and `running_total` (sum of all days so far in 2024).

*Expected result: 25 rows · columns: `order_date`, `daily_revenue`, `running_total`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q5. Salary vs department average  ·  🟢 easy

Show `name`, `dept_id`, `salary`, `dept_avg` (2 decimals) and `diff` = salary − dept_avg (2 decimals) for employees with a department.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `dept_avg`, `diff`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q6. Share of department payroll  ·  🟢 easy

Show `name`, `dept_id`, `salary` and `pct_of_dept` = salary as % of the department's total salary, rounded to 1 decimal. Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `pct_of_dept`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q7. Days since the previous order (LAG)  ·  🟡 medium

For each customer's orders (each order once), show `customer_id`, `order_id`, `order_date`, `prev_order_date` and `days_since_prev` (whole days). The first order of a customer has NULLs.

*Expected result: 104 rows · columns: `customer_id`, `order_id`, `order_date`, `prev_order_date`, `days_since_prev`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q8. Month-over-month change  ·  🔴 hard

Monthly revenue of completed orders (each order once). Show `month` (first day of the month), `revenue`, `prev_revenue`, and `pct_change` = (revenue − prev) / prev × 100, 1 decimal.

*Expected result: 27 rows · columns: `month`, `revenue`, `prev_revenue`, `pct_change`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q9. 3-month moving average  ·  🟡 medium

Same monthly revenue. Show `month`, `revenue`, and `moving_avg_3` = average of this month and the 2 months before it (by row), 2 decimals.

*Expected result: 27 rows · columns: `month`, `revenue`, `moving_avg_3`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q10. Salary quartiles (NTILE)  ·  🟢 easy

Split all employees into 4 groups by salary (highest first): `name`, `salary`, `quartile` (1–4). Break ties by `emp_id`.

*Expected result: 30 rows · columns: `name`, `salary`, `quartile`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q11. Top earner on every row (FIRST_VALUE)  ·  🟡 medium

Show `name`, `dept_id`, `salary` and `top_earner` = the name of the highest-paid person in the same department (ties: first by name). Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `top_earner`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q12. Latest order per customer  ·  🟡 medium

For each customer, show their latest order: `customer_id`, `order_id`, `order_date`, `amount`. If two orders are on the same day, take the bigger `order_id`. One row per customer.

*Expected result: 32 rows · columns: `customer_id`, `order_id`, `order_date`, `amount`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q13. ROWS vs RANGE (ties in a running total)  ·  🔴 hard

Department 2 employees ordered by salary (low → high). Show `name`, `salary`, `rows_total` (frame ROWS ... CURRENT ROW, ties broken by emp_id) and `range_total` (frame RANGE ... CURRENT ROW). Why are they different for the 90,000 tie?

*Expected result: 6 rows · columns: `name`, `salary`, `rows_total`, `range_total`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()